# Cross-Component Interaction

How attention and MLP interact within a layer: reinforcing vs competing,
logit agreement, contribution to prediction, residual mid-point analysis.

## Why This Matters

Cross-component component interaction measures how information transfers between different parts of the model. Understanding cross-component interactions reveals the collaborative computation that produces emergent model capabilities.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.cross_component_interaction import (
    attn_mlp_cosine, attn_mlp_logit_agreement,
    component_contribution_to_prediction, residual_mid_analysis,
    cross_component_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Attention-MLP Cosine

Are attention and MLP reinforcing or competing?

In [ ]:
for layer in range(4):
    result = attn_mlp_cosine(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer}: cos={result['cosine']:.4f} [{result['relationship'].upper()}] "
          f"attn={result['attn_norm']:.4f}, mlp={result['mlp_norm']:.4f}")

## Logit Agreement

Do attention and MLP promote the same tokens?

In [ ]:
for layer in range(4):
    result = attn_mlp_logit_agreement(model, tokens, layer=layer, position=-1, top_k=5)
    print(f"  Layer {layer}: overlap={result['n_overlap']}/5 "
          f"({result['agreement_fraction']:.0%}), shared={result['overlap']}")

## Contribution to Prediction

In [ ]:
for layer in range(4):
    result = component_contribution_to_prediction(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer}: attn={result['attn_contribution']:+.4f}, "
          f"mlp={result['mlp_contribution']:+.4f} -> token {result['predicted_token']}")

## Residual Mid-Point

In [ ]:
for layer in range(4):
    result = residual_mid_analysis(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer}: pre={result['pre_norm']:.4f} -> mid={result['mid_norm']:.4f} "
          f"-> post={result['post_norm']:.4f}")
    print(f"           pre-mid cos={result['pre_mid_cosine']:.4f}, "
          f"mid-post cos={result['mid_post_cosine']:.4f}")

## Summary

In [ ]:
result = cross_component_summary(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: cos={p['cosine']:.4f} [{p['relationship']}], "
          f"agree={p['agreement']:.0%}")